# Per-ROI activity (mean ΔF/F) and spatial information

本 notebook 提取自 `PlaceCell_analysis_8.10_ShuffleUpdate.ipynb`，对**所有 ROI** 直接计算：

1. **每个 ROI 的 activity**：平均 ΔF/F（deconvolved transients 的时间均值）。
2. **每个 ROI 的 spatial information**：Skaggs 空间信息经 shuffle（1000 次）均值归一化后的值。
3. **交互式 ROI 查看器**（notebook 末尾）：加载 suite2p 的 meanImg 与 ROI mask，mask 按 activity / SI 着色为热图，点击任意 ROI 显示其 F（原始荧光）与 deconvolved trace 对比；另附可选的单文件 HTML 导出。

结果保存为 `outputs/roi_mean_dff_slice{slice_no}.npy` 与 `outputs/roi_spatial_info_slice{slice_no}.npy`（数组长度 = ROI 数量，与 suite2p ROI 索引一一对应）。

所需输入：
- `outputs/calcium.nwb`（deconvolved calcium traces）
- `outputs/behavior.nwb`（位置信息）
- `suite2p/plane0/ops.npy`、`stat.npy`、`F.npy`（仅 viewer 部分需要）

注：完全沉默的 ROI（activity 全为 0）的 spatial information 为 NaN，属预期现象。

# Imports & parameters

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

import pynapple as nap
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from joblib import Parallel, delayed
import joblib
from contextlib import contextmanager
from pathlib import Path

# —— 路径与参数 ——
source = Path(r"F:\Code\codex\PlaceCell_snRNAseq\figures\PlaceCells\03_SharedData\Data\CalciumImaging\A77")
output_path = source / "outputs"

add_frames = True
slice_no = 1   # starts from 1

frame_rate = 30
bin_num = 80

# —— spatial information shuffle 参数 ——
n_jobs  = 18
n_bins  = 80
nsample = 1000
window  = 50

# Load the data

加载钙信号（deconvolved transients）与位置信息，并将位置插值到成像时间点上，得到 `Filtered_Position`。

In [ ]:
calcium_path = source / "outputs" / "calcium.nwb"   # 含有calcium.nwb的文件夹
behavior_path = source / "outputs" / "behavior.nwb"  # 行为学数据.nwb

behavior_data = nap.load_file(behavior_path)
calcium_data = nap.load_file(calcium_path)

position = behavior_data['SpatialSeries']
transients = calcium_data["Deconvolved"]

# 位置缩放（与原分析保持一致）
position = nap.Tsd(
    t=position.times(),
    d=np.array(position) * 1,
    time_units='s'
)

print(calcium_data)
print(behavior_data)

In [ ]:
# 可选：在块间插入帧（与原分析保持一致）
if add_frames:
    original_transients = calcium_data["Deconvolved"]
    original_data = original_transients.values
    original_times = original_transients.times()
    dt = np.diff(original_times).mean()  # 帧间隔 (约 0.0333 秒)

    # 参数设置
    chunk_size = 3600          # 每个块3600帧
    insert_frames = 3          # 每个块前插入3帧
    n_chunks = 10              # 总块数 (36000/3600=10)
    total_inserts = insert_frames * n_chunks  # 总插入30帧

    # 预分配内存（精确尺寸）
    new_data = np.zeros((original_data.shape[0] + total_inserts, original_data.shape[1]), dtype=original_data.dtype)
    new_times = np.zeros(original_data.shape[0] + total_inserts)

    current_new_idx = 0

    for chunk in range(n_chunks):
        # === 插入3帧 ===
        if chunk == 0:
            insert_values = np.tile(original_data[0], (insert_frames, 1))
            insert_times = np.linspace(0, insert_frames*dt, insert_frames, endpoint=False)
        else:
            prev_last = original_data[chunk*chunk_size - 1]
            curr_first = original_data[chunk*chunk_size]
            insert_values = np.linspace(prev_last, curr_first, insert_frames+2)[1:-1]
            insert_times = np.linspace(
                start=current_new_time + dt,
                stop=current_new_time + (insert_frames+1)*dt,
                num=insert_frames,
                endpoint=False
            )

        assert len(insert_times) == insert_frames

        new_data[current_new_idx:current_new_idx+insert_frames] = insert_values
        new_times[current_new_idx:current_new_idx+insert_frames] = insert_times
        current_new_idx += insert_frames

        # === 写入当前块数据 ===
        chunk_start = chunk * chunk_size
        chunk_end = chunk_start + chunk_size
        chunk_times = new_times[current_new_idx-1] + dt + np.arange(chunk_size)*dt

        new_data[current_new_idx:current_new_idx+chunk_size] = original_data[chunk_start:chunk_end]
        new_times[current_new_idx:current_new_idx+chunk_size] = chunk_times
        current_new_idx += chunk_size
        current_new_time = new_times[current_new_idx-1]

    assert len(new_times) == 36030
    assert new_data.shape[0] == 36030

    transients = nap.TsdFrame(
        t=new_times,
        d=new_data,
        time_units='s',
        columns=original_transients.columns
    )

print(f"transients shape: {transients.shape}")

In [ ]:
# 去掉重复/非递增时间戳，保证严格递增
t = position.times()
d = position.values
keep = np.r_[True, np.diff(t) > 0]
position = nap.Tsd(
    t=t[keep],
    d=d[keep],
    time_units='s'
)

# 将位置插值到成像时间点
interp_func = interp1d(
    position.times(),
    position.values,
    fill_value=(position.values[0], position.values[-1]),
    bounds_error=False
)
tmp = interp_func(transients.times())
Filtered_Position = nap.Tsd(
    t=transients.times(),
    d=tmp,
    time_units='s'
)
print(Filtered_Position)

# Helper functions

`shuffle`：将活动序列按段打乱（保持均值与总和不变）；

`compute_spatial_information_with_shuffle`：计算 Skaggs 空间信息，并用 `nsample` 次 shuffle 的均值做归一化。

In [ ]:
def shuffle(vector, nsample, window):

    """
    Efficiently shuffles the input vector by splitting it into segments and shuffling these segments without replacement,
    ensuring that the mean and total sum of the sequence remain unchanged after shuffling.

    Parameters:
    - vector: One-dimensional numpy array
    - nsample: Number of shuffles
    - window: Size of each segment

    Returns:
    - vectors: Two-dimensional numpy array with shape (nsample, len(vector))
    """
    vector = np.asarray(vector)
    if vector.ndim != 1:
        raise ValueError("Input vector must be one-dimensional.")
    if not isinstance(nsample, int) or nsample <= 0:
        raise ValueError("nsample must be a positive integer.")
    if not isinstance(window, int) or window <= 0:
        raise ValueError("window must be a positive integer.")

    n = len(vector)
    nsegment = n // window
    remainder = n % window

    if nsegment == 0:
        raise ValueError("Window size is larger than the length of the vector.")
    segments = vector[:nsegment * window].reshape(nsegment, window)

    # Preallocate the shuffled vectors
    shuffled_vectors = np.empty((nsample, nsegment, window), dtype=vector.dtype)

    for i in range(nsample):
        rng = np.random.RandomState(seed=i)  # 固定种子，基于 shuffle 索引
        perm = rng.permutation(nsegment)     # 生成确定性的随机排列
        shuffled_vectors[i] = segments[perm]

    shuffled_vectors = shuffled_vectors.reshape(nsample, -1)

    # If there is a remainder, append the remaining part of the original vector to each shuffled sample
    if remainder > 0:
        remainder_part = vector[nsegment * window:]
        shuffled_vectors = np.hstack([shuffled_vectors, np.tile(remainder_part, (nsample, 1))])
        shuffled_vectors = shuffled_vectors[:, :n]

    return shuffled_vectors


def compute_spatial_information_with_shuffle(neuron_idx, activity, positions, tuning_curves, n_bins=80, nsample=1000, window=50):
    """
    计算归一化的空间信息，使用优化后的打乱函数。

    参数：
    - activity: 一维numpy数组，神经元活动数据
    - positions: 一维numpy数组，位置数据
    - n_bins: 空间区间数量
    - nsample: 打乱次数
    - window: 每个段的大小

    返回：
    - normalized_spatial_info: 归一化的空间信息
    """

    # 将位置数据分到不同的空间区间，设置 right=True 以确保边界值被正确分配
    position_bins = np.linspace(np.min(positions), np.max(positions), n_bins + 1)
    bin_indices = np.digitize(positions, position_bins, right=True) - 1  # 得到每个位置所在的区间

    # 确保 bin_indices 在 0 到 n_bins-1 之间
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    # 计算每个空间区间的时间占比 p_i
    p = np.array([np.sum(bin_indices == i) / len(bin_indices) for i in range(n_bins)])  # 位置分布的概率
    a = np.mean(activity)  # 总体平均活动

    # 计算每个区间的平均活动 a_i，处理空区间
    a_i = tuning_curves[neuron_idx]

    # 防止 a_i 为零，添加一个小的 epsilon
    epsilon = 1e-10
    a_i = np.where(a_i == 0, epsilon, a_i)

    # 计算互信息量 I = sum(p_i * (a_i / a) * log2(a_i / a))
    spatial_info_real = np.sum(p * (a_i / a) * np.log2(a_i / a))

    # Shuffle activity using optimized_shuffle
    shuffled_activity_vectors = shuffle(activity, nsample, window)

    # Compute spatial information for each shuffled vector
    spatial_info_shuffled = np.empty(nsample)

    for i in range(nsample):
        shuffled_activity = shuffled_activity_vectors[i]

        # 计算打乱后活动数据的 a_i
        a_i_shuffled = np.array([np.mean(shuffled_activity[bin_indices == j]) if p[j] > 0 else a for j in range(n_bins)])
        a_i_shuffled = np.where(a_i_shuffled == 0, epsilon, a_i_shuffled)

        # 计算打乱后空间信息
        spatial_info_shuffled[i] = np.sum(p * (a_i_shuffled / a) * np.log2(a_i_shuffled / a))

    # 计算打乱后的空间信息的均值
    mean_shuffled_info = np.mean(spatial_info_shuffled)

    # 计算归一化空间信息
    if mean_shuffled_info == 0:
        normalized_spatial_info = np.nan  # 避免除以零
    else:
        normalized_spatial_info = spatial_info_real / mean_shuffled_info

    return normalized_spatial_info

# Per-ROI activity (mean ΔF/F)

计算每个 ROI 的平均 ΔF/F（deconvolved transients 的时间均值）。

In [ ]:
# 每个 ROI 的平均 ΔF/F
mean_dff = np.asarray(transients).mean(axis=0)   # shape: (N_rois,)
print(f"ROI 数量: {len(mean_dff)}")
print(f"mean ΔF/F: median = {np.median(mean_dff):.4f}, "
      f"min = {np.min(mean_dff):.4f}, max = {np.max(mean_dff):.4f}")

plt.figure()
plt.hist(mean_dff, bins=100)
plt.xlabel('Mean ΔF/F')
plt.ylabel('Number of ROIs')
plt.title('Per-ROI mean ΔF/F distribution')
plt.show()

# Per-ROI spatial information (shuffle-normalized)

对每个 ROI 并行计算归一化空间信息：真实空间信息 / shuffle 均值。

In [ ]:
# —— 数据准备（关键：tuning_curves 需为 (N_neurons, n_bins)）——
temp_all_tuning_curves = nap.compute_1d_tuning_curves_continuous(transients, feature=Filtered_Position, nb_bins=n_bins)
transients_np = np.asarray(transients)                 # (T, N)
positions_np  = np.asarray(Filtered_Position)          # (T,)
tuning_curves_full = np.asarray(temp_all_tuning_curves).T  # (N, n_bins)

# 可选一致性检查
assert transients_np.shape[0] == positions_np.shape[0]
assert tuning_curves_full.shape[1] == n_bins

def process_neuron(neuron_idx):
    activity = transients_np[:, neuron_idx]
    return compute_spatial_information_with_shuffle(
        neuron_idx, activity, positions_np, tuning_curves_full,
        n_bins=n_bins, nsample=nsample, window=window
    )

# —— joblib × tqdm 集成 ——
@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_cb = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_cb
        tqdm_object.close()

n_rois = transients_np.shape[1]

if __name__ == "__main__":  # Windows 建议加
    with tqdm_joblib(tqdm(total=n_rois, desc="Spatial info (shuffle)", unit="neuron")):
        all_cells_si = Parallel(n_jobs=n_jobs, prefer="processes")(
            delayed(process_neuron)(idx) for idx in range(n_rois)
        )
    all_cells_si = np.asarray(all_cells_si)

In [ ]:
valid = all_cells_si[np.isfinite(all_cells_si)]
print(f"有效 ROI: {len(valid)} / {len(all_cells_si)} (沉默 ROI 为 NaN)")
print(f"normalized SI: median = {np.median(valid):.3f}, "
      f"min = {np.min(valid):.3f}, max = {np.max(valid):.3f}")

plt.figure()
plt.hist(valid, bins=100)
plt.xlabel('Normalized Spatial Information')
plt.ylabel('Number of ROIs')
plt.title('Per-ROI spatial information distribution')
plt.show()

# Save results

保存每个 ROI 的结果（数组长度 = ROI 数量，索引与 suite2p ROI 一一对应）。

In [ ]:
np.save(output_path / f"roi_mean_dff_slice{slice_no}.npy", mean_dff)
np.save(output_path / f"roi_spatial_info_slice{slice_no}.npy", all_cells_si)

print(f"Saved: {output_path / f'roi_mean_dff_slice{slice_no}.npy'}")
print(f"Saved: {output_path / f'roi_spatial_info_slice{slice_no}.npy'}")

# Interactive ROI viewer (suite2p)

加载 suite2p 的 `ops.npy` / `stat.npy` / `F.npy`：

- **左面板**：meanImg 灰度底图 + ROI mask 热图叠加（NaN 处透明），Dropdown 切换着色量（Mean activity / Spatial information / None），色阶取 1–99 percentile 防止 outlier 压制
- **点击某个 ROI** → 右面板画出该 ROI 的 **F（原始荧光，绿）** 与 **deconvolved（红）** trace（双 y 轴），标题显示 ROI 索引、activity 和 SI
- 依赖 `plotly` FigureWidget + `ipywidgets`（ipywidgets ≥ 8）。若 widget 无法显示，检查 kernel 环境中 ipywidgets 是否正常

注意：`F.npy` 与 deconvolved 均为原始 36000 帧，二者时间轴一致；viewer 中不使用插帧后的 `transients`（36030 帧）。

In [ ]:
suite2p_path = source / "suite2p" / "plane0"
ops = np.load(suite2p_path / "ops.npy", allow_pickle=True).item()
stat = np.load(suite2p_path / "stat.npy", allow_pickle=True)
F = np.load(suite2p_path / "F.npy")          # (n_rois, n_frames) 原始荧光
mean_img = ops['meanImg']

# 原始 deconvolved traces（与 F.npy 帧数一致；不使用插帧后的 36030 帧版本）
deconv = np.asarray(calcium_data["Deconvolved"])
t_trace = np.arange(F.shape[1]) / frame_rate

assert F.shape[0] == len(stat), "F.npy 与 stat.npy 的 ROI 数量不一致"
assert F.shape[0] == deconv.shape[1], "F.npy 与 calcium.nwb 的 ROI 数量不一致"
assert deconv.shape[0] == F.shape[1], "F.npy 与 deconvolved 的帧数不一致"

Ly, Lx = mean_img.shape

# label_img：每个像素填入其所属 ROI 的索引（-1 = 背景），用于点击查找
label_img = np.full((Ly, Lx), -1, dtype=np.int32)
for i, s in enumerate(stat):
    yp = np.asarray(s['ypix'], dtype=int)
    xp = np.asarray(s['xpix'], dtype=int)
    valid = (yp >= 0) & (yp < Ly) & (xp >= 0) & (xp < Lx)
    label_img[yp[valid], xp[valid]] = i
centroids = np.array([s['med'] for s in stat], dtype=float)   # (N, 2): y, x

print(f"meanImg: {mean_img.shape} | ROIs: {F.shape[0]} | frames: {F.shape[1]}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

color_options = {'Mean activity (ΔF/F)': mean_dff, 'Spatial information': all_cells_si}

def make_value_img(values):
    vimg = np.full((Ly, Lx), np.nan)
    m = label_img >= 0
    vimg[m] = values[label_img[m]]
    return vimg

def value_range(v):
    valid = v[np.isfinite(v)]
    return float(np.percentile(valid, 1)), float(np.percentile(valid, 99))

fig = make_subplots(
    rows=1, cols=2, column_widths=[0.42, 0.58],
    specs=[[{"secondary_y": False}, {"secondary_y": True}]],
    subplot_titles=("ROI map (click a ROI)", "Trace"),
)

# 左：meanImg 灰度底图 + ROI 值热图叠加（NaN 处透明，露出底图）
fig.add_trace(go.Heatmap(z=mean_img, colorscale='gray', showscale=False, hoverinfo='skip'),
              row=1, col=1)
fig.add_trace(go.Heatmap(
    z=make_value_img(mean_dff), colorscale='Viridis', opacity=0.75,
    colorbar=dict(title='Mean activity (ΔF/F)', x=0.40, len=0.9),
    hovertemplate='x=%{x}, y=%{y}, value=%{z:.3g}<extra></extra>'),
    row=1, col=1)
fig.add_trace(go.Scattergl(x=[], y=[], mode='markers',
                           marker=dict(size=12, color='red', symbol='cross-thin-open'),
                           showlegend=False),
              row=1, col=1)

# 右：F（左轴）+ deconvolved（右轴）
fig.add_trace(go.Scattergl(x=t_trace, y=[np.nan] * len(t_trace), name='F (raw)',
                           line=dict(color='green', width=1), opacity=0.8),
              row=1, col=2, secondary_y=False)
fig.add_trace(go.Scattergl(x=t_trace, y=[np.nan] * len(t_trace), name='deconvolved',
                           line=dict(color='firebrick', width=1)),
              row=1, col=2, secondary_y=True)

fw = go.FigureWidget(fig)
fw.update_yaxes(autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fw.update_yaxes(title_text='F', row=1, col=2, secondary_y=False)
fw.update_yaxes(title_text='deconvolved', row=1, col=2, secondary_y=True)
fw.update_layout(width=1350, height=600, title=dict(text='Click a ROI'),
                 margin=dict(l=40, r=40, t=70, b=40),
                 legend=dict(orientation='h', y=1.06, x=0.45))

# 初始色阶：1–99 percentile
zmin, zmax = value_range(mean_dff)
fw.data[1].zmin, fw.data[1].zmax = zmin, zmax

# trace 顺序：0 meanImg, 1 overlay, 2 选中标记, 3 F, 4 deconv
def handle_click(trace, points, state):
    if not points.xs:
        return
    x, y = int(round(points.xs[0])), int(round(points.ys[0]))
    roi = -1
    if 0 <= y < Ly and 0 <= x < Lx:
        roi = int(label_img[y, x])
    if roi < 0:
        # 兜底：6 px 内最近的 ROI 质心
        d2 = (centroids[:, 1] - x) ** 2 + (centroids[:, 0] - y) ** 2
        j = int(np.argmin(d2))
        if d2[j] <= 36:
            roi = j
    if roi < 0:
        return
    si_str = f"{all_cells_si[roi]:.3f}" if np.isfinite(all_cells_si[roi]) else "NaN"
    with fw.batch_update():
        fw.data[3].y = F[roi]
        fw.data[4].y = deconv[:, roi]
        fw.data[2].x = [centroids[roi, 1]]
        fw.data[2].y = [centroids[roi, 0]]
        fw.layout.title.text = f"ROI {roi} | mean ΔF/F = {mean_dff[roi]:.4f} | SI = {si_str}"

fw.data[1].on_click(handle_click)

dd = widgets.Dropdown(
    options=['Mean activity (ΔF/F)', 'Spatial information', 'None'],
    value='Mean activity (ΔF/F)',
    description='Color by:',
)

def on_change(change):
    key = change['new']
    with fw.batch_update():
        if key == 'None':
            fw.data[1].z = np.full((Ly, Lx), np.nan)
            fw.data[1].zmin = None
            fw.data[1].zmax = None
            fw.data[1].colorbar.title.text = ''
        else:
            v = color_options[key]
            fw.data[1].z = make_value_img(v)
            fw.data[1].zmin, fw.data[1].zmax = value_range(v)
            fw.data[1].colorbar.title.text = key

dd.observe(on_change, names='value')
display(dd, fw)

# （可选）导出单文件 HTML 查看器

将 meanImg + ROI 散点 + 抽稀 trace 打包为单个 HTML，双击即可打开、便于分享：

- trace 按 `ds` 帧抽稀（30 Hz → 2.5 Hz），仅用于快速查看
- plotly.js 从 CDN 加载，**打开时需要联网**
- `export_subset = None` 导出全部 ROI（文件约几十 MB）；可传入 ROI 索引列表只导出部分以减小体积

In [ ]:
# —— 可选：导出单文件 HTML 查看器 ——
import base64, io
import plotly.graph_objects as go

export_subset = None   # None = 全部 ROI；或传入索引列表，如 [10, 20, 30]
ds = 12                # 抽稀步长（30 Hz -> 2.5 Hz）
html_path = output_path / f"roi_viewer_slice{slice_no}.html"

if export_subset is None:
    roi_sel = np.arange(F.shape[0])
else:
    roi_sel = np.asarray(export_subset, dtype=int)

# 抽稀 trace（float32，base64 二进制嵌入，比 JSON 更小更快）
F_ds = np.ascontiguousarray(F[roi_sel, ::ds].astype(np.float32))
D_ds = np.ascontiguousarray(deconv.T[roi_sel, ::ds].astype(np.float32))
n_tr = F_ds.shape[1]
b64_F = base64.b64encode(F_ds.tobytes()).decode()
b64_D = base64.b64encode(D_ds.tobytes()).decode()

# meanImg -> base64 PNG 底图
buf = io.BytesIO()
plt.imsave(buf, mean_img, cmap='gray', format='png')
b64_img = base64.b64encode(buf.getvalue()).decode()

xs = centroids[roi_sel, 1]
ys = centroids[roi_sel, 0]
act = mean_dff[roi_sel]
si = all_cells_si[roi_sel]
si_list = [None if not np.isfinite(v) else float(v) for v in si]   # JSON 不支持 NaN -> null
cd = [[float(a), s] for a, s in zip(act, si_list)]

fig2 = go.Figure()
fig2.add_trace(go.Scattergl(
    x=xs, y=ys, mode='markers',
    text=roi_sel,
    marker=dict(size=5, color=act.tolist(), colorscale='Viridis',
                colorbar=dict(title='Mean activity')),
    customdata=cd,
    hovertemplate='ROI %{text}<br>activity=%{customdata[0]:.4f}<br>SI=%{customdata[1]:.3g}<extra></extra>',
))

fig2.update_layout(
    title='Click a ROI (exported viewer)',
    width=900, height=900,
    xaxis=dict(range=[0, Lx], showgrid=False),
    yaxis=dict(range=[Ly, 0], showgrid=False),
    images=[dict(source=f"data:image/png;base64,{b64_img}",
                 xref='x', yref='y', x=0, y=Ly, sizex=Lx, sizey=Ly,
                 sizing='stretch', opacity=1, layer='below')],
    updatemenus=[dict(
        type='buttons', direction='right', x=0, y=1.08,
        buttons=[
            dict(label='Color: Mean activity', method='restyle',
                 args=[{'marker.color': [act.tolist()],
                        'marker.colorbar.title.text': 'Mean activity'}, [0]]),
            dict(label='Color: Spatial info', method='restyle',
                 args=[{'marker.color': [si_list],
                        'marker.colorbar.title.text': 'SI'}, [0]]),
        ],
    )],
)

# 点击 ROI -> 在页面下方创建 trace 面板并绘制 F / deconvolved
# 注意：{plot_id} 是 plotly 的占位符，write_html 时自动替换为 div id，勿改动
post_script = """
(function () {
  var gd = document.getElementById('{plot_id}');
  if (!gd) return;
  var N_TR = @N_TR@, DS = @DS@, FR = @FR@;
  var X_TR = new Array(N_TR);
  for (var i = 0; i < N_TR; i++) X_TR[i] = i * DS / FR;
  function b64ToF32(b64) {
    var bin = atob(b64), b = new Uint8Array(bin.length);
    for (var i = 0; i < bin.length; i++) b[i] = bin.charCodeAt(i);
    return new Float32Array(b.buffer);
  }
  var F32 = b64ToF32('@B64F@');
  var D32 = b64ToF32('@B64D@');
  var traceDiv = document.createElement('div');
  traceDiv.style.width = '900px';
  traceDiv.style.height = '420px';
  traceDiv.style.marginTop = '10px';
  gd.parentNode.appendChild(traceDiv);
  var baseLayout = { title: 'Click a ROI to view its trace', height: 420 };
  Plotly.newPlot(traceDiv, [
    { x: X_TR, y: [], name: 'F (raw)', line: { color: 'green', width: 1 } },
    { x: X_TR, y: [], name: 'deconvolved', line: { color: 'firebrick', width: 1 } }
  ], baseLayout);
  gd.on('plotly_click', function (ev) {
    if (!ev.points || !ev.points.length) return;
    var idx = ev.points[0].pointIndex;
    var start = idx * N_TR;
    var roi = gd.data[0].text ? gd.data[0].text[idx] : idx;
    var cd = gd.data[0].customdata ? gd.data[0].customdata[idx] : [NaN, NaN];
    var title = 'ROI ' + roi + ' | activity = ' + cd[0].toFixed(4) +
      ' | SI = ' + (cd[1] === null ? 'NaN' : cd[1].toFixed(3));
    Plotly.react(traceDiv, [
      { x: X_TR, y: Array.from(F32.subarray(start, start + N_TR)), name: 'F (raw)', line: { color: 'green', width: 1 } },
      { x: X_TR, y: Array.from(D32.subarray(start, start + N_TR)), name: 'deconvolved', line: { color: 'firebrick', width: 1 } }
    ], { title: title, height: 420 });
  });
})();
"""
post_script = (post_script
    .replace('@N_TR@', str(n_tr))
    .replace('@DS@', str(ds))
    .replace('@FR@', str(float(frame_rate)))
    .replace('@B64F@', b64_F)
    .replace('@B64D@', b64_D))

fig2.write_html(str(html_path), include_plotlyjs='cdn', post_script=post_script)
print(f"HTML viewer saved: {html_path}")
print(f"file size: {html_path.stat().st_size / 1e6:.1f} MB | exported ROIs: {len(roi_sel)}")